# Distributed Machine Learning on Banking Data
### Hadoop • Hive • Apache Spark • Spark ML • Spark Streaming

This notebook implements all 5 parts of the project using **PySpark** running in **local (single-node) mode inside Google Colab**, which simulates a distributed cluster:

| Part | Objective | Tool |
|---|---|---|
| 1 | Data Storage & Querying | Spark SQL with Hive support (acts as Hive on top of Hadoop-style storage) |
| 2 | Exploratory Data Analysis | Spark DataFrame API |
| 3 | Predictive Modeling | Spark ML (Pipelines, Logistic Regression, Random Forest) |
| 4 | Real-Time Transaction Analysis | Spark Structured Streaming |
| 5 | Data Parallelism | Partitioning, caching, broadcast joins |

**Note on environment:** Google Colab does not give you a real multi-node Hadoop cluster. The standard, accepted approach for a project like this is to run **Spark in local mode with Hive support enabled** (`enableHiveSupport()`), which uses Spark's built-in Hive metastore and HiveQL engine — this is the same query engine and SQL dialect a real Hadoop/Hive cluster would use, just running on one machine instead of many. This is explained further in the accompanying reflective summary document, and is worth saying explicitly in your video (see script notes at the bottom of this notebook).

Upload `bank.csv` to the Colab file browser (left sidebar) before running, or mount Google Drive.


In [ ]:
# ============================================================
# SETUP — install PySpark and start a Spark session with Hive support
# ============================================================
!pip install -q pyspark==3.5.1

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName("BankingDistributedML")
    .config("spark.sql.warehouse.dir", "/content/spark-warehouse")
    .enableHiveSupport()
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)
spark


In [ ]:
# Upload bank.csv here if not already present (Colab file upload widget)
from google.colab import files
import os

if not os.path.exists("bank.csv"):
    uploaded = files.upload()  # select bank.csv from your machine


---
## Part 1 — Data Storage & Management with Hadoop / Hive

**Goal:** Simulate how a bank stores and queries large transaction/customer datasets using a Hive-style warehouse on top of distributed (HDFS-like) storage. We create a managed Hive database and table, load the raw CSV into it, and run HiveQL queries.


In [ ]:
# 1.1 Load the raw CSV into a Spark DataFrame with an explicit schema
# (explicit schema = good practice; avoids costly schema inference on large data)

schema = StructType([
    StructField("age", IntegerType(), True),
    StructField("job", StringType(), True),
    StructField("marital", StringType(), True),
    StructField("education", StringType(), True),
    StructField("default", StringType(), True),
    StructField("balance", IntegerType(), True),
    StructField("housing", StringType(), True),
    StructField("loan", StringType(), True),
    StructField("contact", StringType(), True),
    StructField("day", IntegerType(), True),
    StructField("month", StringType(), True),
    StructField("duration", IntegerType(), True),
    StructField("campaign", IntegerType(), True),
    StructField("pdays", IntegerType(), True),
    StructField("previous", IntegerType(), True),
    StructField("poutcome", StringType(), True),
    StructField("y", StringType(), True),
])

df_raw = spark.read.csv("bank.csv", header=True, schema=schema)
df_raw.printSchema()
df_raw.show(5)
print("Row count:", df_raw.count())


In [ ]:
# 1.2 Create a Hive database + managed table and load the data into it
spark.sql("CREATE DATABASE IF NOT EXISTS banking_dw")
spark.sql("USE banking_dw")

df_raw.write.mode("overwrite").saveAsTable("banking_dw.customer_transactions")

spark.sql("SHOW TABLES IN banking_dw").show()
spark.sql("DESCRIBE FORMATTED banking_dw.customer_transactions").show(50, truncate=False)


In [ ]:
# 1.3 Run HiveQL queries directly against the managed table
# -- Query A: subscription rate overall
spark.sql("""
    SELECT y, COUNT(*) AS n_customers,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM banking_dw.customer_transactions
    GROUP BY y
""").show()

# -- Query B: average balance and subscription rate by job type
spark.sql("""
    SELECT job,
           COUNT(*) AS n_customers,
           ROUND(AVG(balance), 2) AS avg_balance,
           ROUND(100.0 * SUM(CASE WHEN y = 'yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS subscribe_rate_pct
    FROM banking_dw.customer_transactions
    GROUP BY job
    ORDER BY subscribe_rate_pct DESC
""").show(20, truncate=False)

# -- Query C: campaign effectiveness by previous outcome
spark.sql("""
    SELECT poutcome,
           COUNT(*) AS contacts,
           ROUND(100.0 * SUM(CASE WHEN y = 'yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS success_rate_pct
    FROM banking_dw.customer_transactions
    GROUP BY poutcome
    ORDER BY success_rate_pct DESC
""").show()


**What this demonstrates:** in a real deployment, `bank.csv` would land in HDFS (e.g. via Sqoop/Flume/batch ingestion), Hive would define the external/managed table schema over that HDFS location, and analysts would query it with HiveQL exactly as above — without needing to know Spark or Python. Spark's Hive integration lets the *same* SQL engine be used for both ad-hoc querying (Hive) and programmatic analysis (Spark), which is why they're paired in real banking data platforms.

---
## Part 2 — Exploratory Data Analysis (EDA) with Spark

**Goal:** Use the distributed Spark DataFrame API (not pandas) to explore the dataset at scale — schema, summary statistics, distributions, correlations, and relationships with the target variable `y`.


In [ ]:
# 2.1 Read back from the Hive table (this is now our source of truth)
df = spark.table("banking_dw.customer_transactions")

# Basic shape & schema
print("Rows:", df.count(), " Columns:", len(df.columns))
df.printSchema()


In [ ]:
# 2.2 Summary statistics for numeric columns
numeric_cols = ["age", "balance", "day", "duration", "campaign", "pdays", "previous"]
df.select(numeric_cols).describe().show()


In [ ]:
# 2.3 Missing / sentinel value checks
# Nulls
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

# 'unknown' categorical values (this dataset encodes missing categoricals as the literal string 'unknown')
cat_cols = ["job", "marital", "education", "default", "housing", "loan", "contact", "month", "poutcome"]
for c in cat_cols:
    n_unknown = df.filter(F.col(c) == "unknown").count()
    if n_unknown > 0:
        print(f"{c}: {n_unknown} rows marked 'unknown' ({round(100*n_unknown/df.count(),1)}%)")


In [ ]:
# 2.4 Class balance of the target variable
df.groupBy("y").count().withColumn(
    "pct", F.round(100 * F.col("count") / df.count(), 2)
).show()
# Note: this is an imbalanced classification problem (~88% 'no' vs ~12% 'yes'),
# which we account for later in Part 3 when choosing evaluation metrics.


In [ ]:
# 2.5 Subscription rate broken down by key categorical features
for c in ["job", "marital", "education", "housing", "loan", "contact", "poutcome"]:
    print(f"\n--- {c} ---")
    df.groupBy(c).agg(
        F.count("*").alias("n"),
        F.round(100 * F.avg(F.when(F.col("y") == "yes", 1).otherwise(0)), 2).alias("subscribe_rate_pct")
    ).orderBy(F.desc("subscribe_rate_pct")).show(truncate=False)


In [ ]:
# 2.6 Numeric feature distributions by target class (mean comparison)
df.groupBy("y").agg(
    *[F.round(F.avg(c), 2).alias(f"avg_{c}") for c in numeric_cols]
).show(truncate=False)


In [ ]:
# 2.7 Correlation matrix for numeric features (Spark's Correlation module)
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

assembler = VectorAssembler(inputCols=numeric_cols, outputCol="features_corr")
vec_df = assembler.transform(df).select("features_corr")
corr_matrix = Correlation.corr(vec_df, "features_corr").head()[0].toArray()

import pandas as pd
corr_pd = pd.DataFrame(corr_matrix, index=numeric_cols, columns=numeric_cols).round(2)
corr_pd


In [ ]:
# 2.8 Visualize a few distributions (aggregate in Spark, plot small result in pandas/matplotlib)
import matplotlib.pyplot as plt

# Age distribution
age_pd = df.select("age").toPandas()
plt.figure(figsize=(7,4))
plt.hist(age_pd["age"], bins=30)
plt.title("Age Distribution")
plt.xlabel("Age"); plt.ylabel("Count")
plt.show()

# Subscription rate by month (in calendar order)
month_order = ["jan","feb","mar","apr","may","jun","jul","aug","sep","oct","nov","dec"]
month_pd = (
    df.groupBy("month")
      .agg(F.round(100*F.avg(F.when(F.col("y")=="yes",1).otherwise(0)),2).alias("rate"))
      .toPandas()
)
month_pd["month"] = pd.Categorical(month_pd["month"], categories=month_order, ordered=True)
month_pd = month_pd.sort_values("month")

plt.figure(figsize=(9,4))
plt.bar(month_pd["month"].astype(str), month_pd["rate"])
plt.title("Subscription Rate by Month")
plt.xlabel("Month"); plt.ylabel("Subscribe Rate (%)")
plt.show()


**Key EDA findings to mention in your video:**
- The target class is imbalanced (~88% *no* vs ~12% *yes*) → plain accuracy is a misleading metric; use AUC/precision/recall (see Part 3).
- `duration` (last call length) is strongly associated with `y`, but it's only known *after* a call happens, so in a real "predict before calling" model it should usually be excluded — worth discussing as a modeling trade-off.
- Customers with `poutcome = success` on a prior campaign convert far more often — history matters.
- Students, retired people, and management-level jobs tend to show higher subscription rates than blue-collar jobs.


---
## Part 3 — Predictive Modeling with Spark ML

**Goal:** Predict whether a customer will subscribe to a term deposit (`y`), using Spark ML's distributed Pipeline API (indexers, encoders, assembler → classifier), trained on a train/test split, and evaluated with metrics appropriate for imbalanced data.


In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

model_df = df

categorical_cols = ["job", "marital", "education", "default", "housing",
                     "loan", "contact", "month", "poutcome"]
numeric_features = ["age", "balance", "day", "duration", "campaign", "pdays", "previous"]

# Index + one-hot encode categoricals
indexers = [StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep") for c in categorical_cols]
encoders = [OneHotEncoder(inputCol=c+"_idx", outputCol=c+"_ohe") for c in categorical_cols]

# Index the label
label_indexer = StringIndexer(inputCol="y", outputCol="label")

# Assemble all features into one vector
feature_cols = [c+"_ohe" for c in categorical_cols] + numeric_features
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")


In [ ]:
# 3.1 Train / test split (stratification approximated via seed + check below)
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)
print("Train rows:", train_df.count(), " Test rows:", test_df.count())
train_df.groupBy("y").count().show()
test_df.groupBy("y").count().show()


In [ ]:
# 3.2 Model A — Logistic Regression pipeline
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=50)
pipeline_lr = Pipeline(stages=indexers + encoders + [label_indexer, assembler, lr])

lr_model = pipeline_lr.fit(train_df)
lr_preds = lr_model.transform(test_df)

evaluator_auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", metricName="f1")
evaluator_precision = MulticlassClassificationEvaluator(labelCol="label", metricName="weightedPrecision")
evaluator_recall = MulticlassClassificationEvaluator(labelCol="label", metricName="weightedRecall")

print("Logistic Regression:")
print("  AUC       :", round(evaluator_auc.evaluate(lr_preds), 4))
print("  Accuracy  :", round(evaluator_acc.evaluate(lr_preds), 4))
print("  F1        :", round(evaluator_f1.evaluate(lr_preds), 4))
print("  Precision :", round(evaluator_precision.evaluate(lr_preds), 4))
print("  Recall    :", round(evaluator_recall.evaluate(lr_preds), 4))


In [ ]:
# 3.3 Model B — Random Forest pipeline (usually stronger on mixed categorical/numeric banking data)
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=100, maxDepth=8, seed=42)
pipeline_rf = Pipeline(stages=indexers + encoders + [label_indexer, assembler, rf])

rf_model = pipeline_rf.fit(train_df)
rf_preds = rf_model.transform(test_df)

print("Random Forest:")
print("  AUC       :", round(evaluator_auc.evaluate(rf_preds), 4))
print("  Accuracy  :", round(evaluator_acc.evaluate(rf_preds), 4))
print("  F1        :", round(evaluator_f1.evaluate(rf_preds), 4))
print("  Precision :", round(evaluator_precision.evaluate(rf_preds), 4))
print("  Recall    :", round(evaluator_recall.evaluate(rf_preds), 4))


In [ ]:
# 3.4 Confusion matrix for the better-performing model
rf_preds.groupBy("label", "prediction").count().orderBy("label", "prediction").show()


In [ ]:
# 3.5 Feature importances from the Random Forest model
rf_stage = rf_model.stages[-1]
importances = rf_stage.featureImportances.toArray()

# Reconstruct expanded feature names for interpretability
import numpy as np
feat_names = []
for c in categorical_cols:
    n_categories = train_df.select(c).distinct().count()
    feat_names += [f"{c}_{i}" for i in range(n_categories)]  # approximate; OHE drops last category
feat_names += numeric_features

imp_pd = pd.DataFrame({"importance": importances[:len(feat_names)]}, index=feat_names[:len(importances)])
imp_pd.sort_values("importance", ascending=False).head(15)


**Modeling notes for your video / write-up:**
- Because `y` is imbalanced, **AUC and F1** are more meaningful than raw accuracy (a model that always predicts "no" would still be ~88% "accurate").
- Random Forest typically outperforms Logistic Regression here because it captures non-linear interactions between categorical banking features (job × education × poutcome, etc.) without manual feature crossing.
- `duration`, `poutcome`, `balance`, and `age` are usually among the strongest predictors — consistent with the EDA findings.
- In production, this Spark ML `Pipeline` (indexers + encoder + assembler + model) is saved as a single artifact and can be applied to new streaming records — which connects directly to Part 4.


---
## Part 4 — Real-Time Transaction Analysis with Spark Streaming

**Goal:** Simulate a real-time feed of banking transactions and process it with **Spark Structured Streaming**, computing rolling/windowed aggregations that a bank would use for live monitoring and fraud/anomaly signals.

**Simulated data:** `bank.csv` was split into 10 shuffled batch files (`transactions_batch_01.csv` … `transactions_batch_10.csv`), each representing a new "arrival" of transactions. In Colab we drip-feed these files one at a time into a folder that Spark watches with `readStream`, which is the standard way to demo Structured Streaming without a live Kafka/socket source.

Upload the provided `streaming_chunks/` folder (10 CSV files) to Colab before running this part, or generate it yourself from `bank.csv` using the helper cell below.


In [ ]:
# 4.1 (Optional) Regenerate the streaming batch files yourself from bank.csv
import pandas as pd, os

os.makedirs("stream_source", exist_ok=True)   # Spark will watch this folder
os.makedirs("stream_incoming", exist_ok=True)  # staging area we drip-feed from

pdf = pd.read_csv("bank.csv").sample(frac=1, random_state=42).reset_index(drop=True)
n_batches = 10
batch_size = len(pdf)//n_batches + 1
for i in range(n_batches):
    chunk = pdf.iloc[i*batch_size:(i+1)*batch_size]
    if len(chunk) == 0:
        continue
    chunk.to_csv(f"stream_incoming/transactions_batch_{i+1:02d}.csv", index=False)

print("Prepared", len(os.listdir("stream_incoming")), "batch files in stream_incoming/")


In [ ]:
# 4.2 Define the streaming schema (streaming sources require an explicit schema — no inference)
streaming_schema = schema  # reuse the schema defined in Part 1

# 4.3 Start a Structured Streaming read on the watched folder
streaming_df = (
    spark.readStream
    .schema(streaming_schema)
    .option("header", True)
    .option("maxFilesPerTrigger", 1)   # process one file ("micro-batch") at a time
    .csv("stream_source")
)

print("Is streaming:", streaming_df.isStreaming)


In [ ]:
# 4.4 Define a live aggregation: rolling subscribe rate & average balance per job type,
# recomputed after every micro-batch (this is the kind of live dashboard metric
# a bank's ops team would watch for campaign monitoring or anomaly detection).

agg_stream = (
    streaming_df
    .groupBy("job")
    .agg(
        F.count("*").alias("transactions_seen"),
        F.round(F.avg("balance"), 2).alias("avg_balance"),
        F.round(100 * F.avg(F.when(F.col("y") == "yes", 1).otherwise(0)), 2).alias("subscribe_rate_pct")
    )
)

query = (
    agg_stream.writeStream
    .outputMode("complete")     # recompute full aggregate each trigger (fits a small demo dataset)
    .format("memory")           # sink to an in-memory table we can query with SQL
    .queryName("live_job_stats")
    .start()
)


In [ ]:
# 4.5 Drip-feed the batch files into the watched folder, one every few seconds,
# and query the in-memory streaming result table as it updates.
import shutil, time

batch_files = sorted(os.listdir("stream_incoming"))
for i, fname in enumerate(batch_files, start=1):
    shutil.copy(f"stream_incoming/{fname}", f"stream_source/{fname}")
    time.sleep(4)  # give Spark time to detect and process the new file
    print(f"--- After ingesting batch {i}/{len(batch_files)}: {fname} ---")
    spark.sql("SELECT * FROM live_job_stats ORDER BY subscribe_rate_pct DESC").show(5, truncate=False)

query.stop()
print("Streaming query stopped.")


In [ ]:
# 4.6 (Optional) A second streaming query demonstrating a windowed alert-style rule:
# flag micro-batches where the incoming average balance is unusually low/high
# (a simplified stand-in for a real-time fraud/anomaly alert).

streaming_df2 = (
    spark.readStream
    .schema(streaming_schema)
    .option("header", True)
    .option("maxFilesPerTrigger", 1)
    .csv("stream_source")   # NOTE: re-run cell 4.3's folder prep / re-drip files if you want to demo this live
)

alert_stream = (
    streaming_df2
    .withColumn("high_value_flag", F.when(F.col("balance") > 5000, 1).otherwise(0))
    .groupBy()
    .agg(
        F.avg("balance").alias("avg_balance"),
        F.sum("high_value_flag").alias("high_value_transactions")
    )
)

# In a real fraud-detection setup you would writeStream to a sink (Kafka topic, dashboard,
# or alerting service) with a foreachBatch() callback that pages an analyst when thresholds are crossed.
print("Example alert-stream definition created (not started, to avoid a second long-running query in this demo).")


**Streaming notes for your video / write-up:**
- `maxFilesPerTrigger=1` forces Spark to treat each uploaded CSV as one micro-batch, mimicking a continuous feed without needing a real message broker (Kafka/Kinesis) for the demo.
- `outputMode("complete")` recomputes the full aggregate table each trigger, which is fine for small aggregates like this; a production system with unbounded state would instead use `update` mode with watermarking.
- The `memory` sink is for demonstration only — production banking systems would sink to Kafka, a data warehouse, or a live dashboard (e.g. Grafana) instead.
- This directly extends Part 3: the same fitted `rf_model` pipeline could be applied inside `foreachBatch` to score each incoming transaction in real time for fraud/propensity scoring.


---
## Part 5 — Efficient Data Handling through Data Parallelism

**Goal:** Demonstrate how Spark's data-parallel execution model (partitioning, caching, broadcast joins) improves processing efficiency on large-scale data, and show measurable before/after evidence.


In [ ]:
# 5.1 Inspect default partitioning
print("Default partitions for df:", df.rdd.getNumPartitions())

# Repartition explicitly (simulating scaling out across more executors/cores)
df_repart = df.repartition(8)
print("Partitions after repartition(8):", df_repart.rdd.getNumPartitions())

# Coalesce down (simulating reducing shuffle/output file count before a write)
df_coalesced = df_repart.coalesce(2)
print("Partitions after coalesce(2):", df_coalesced.rdd.getNumPartitions())


In [ ]:
# 5.2 Measure the effect of caching on a repeated, expensive aggregation
import time

expensive_df = df.withColumn("balance_bucket", F.floor(F.col("balance") / 100))

# --- without caching: recomputed from scratch on every action ---
t0 = time.time()
for _ in range(3):
    expensive_df.groupBy("balance_bucket").agg(F.avg("age"), F.count("*")).collect()
t_uncached = time.time() - t0

# --- with caching: computed once, reused after ---
expensive_df_cached = expensive_df.cache()
expensive_df_cached.count()  # materialize the cache

t0 = time.time()
for _ in range(3):
    expensive_df_cached.groupBy("balance_bucket").agg(F.avg("age"), F.count("*")).collect()
t_cached = time.time() - t0

print(f"3x repeated aggregation, uncached: {t_uncached:.3f}s")
print(f"3x repeated aggregation, cached  : {t_cached:.3f}s")
print(f"Speedup: {t_uncached / max(t_cached, 1e-6):.2f}x")

expensive_df_cached.unpersist()


In [ ]:
# 5.3 Broadcast join demonstration
# Small lookup table (e.g. a job-category-to-segment mapping a bank marketing team maintains)
job_segment_map = spark.createDataFrame([
    ("management", "Premium"), ("technician", "Standard"), ("entrepreneur", "Premium"),
    ("blue-collar", "Standard"), ("unknown", "Standard"), ("retired", "Premium"),
    ("admin.", "Standard"), ("services", "Standard"), ("self-employed", "Premium"),
    ("unemployed", "Basic"), ("housemaid", "Standard"), ("student", "Basic"),
], ["job", "segment"])

from pyspark.sql.functions import broadcast

# Explicit broadcast hint: ships the small table to every executor instead of shuffling
# the large customer table across the network — a key data-parallelism optimization.
joined = df.join(broadcast(job_segment_map), on="job", how="left")
joined.groupBy("segment").agg(
    F.count("*").alias("n_customers"),
    F.round(100*F.avg(F.when(F.col("y")=="yes",1).otherwise(0)),2).alias("subscribe_rate_pct")
).show()

joined.select("job", "segment").explain()  # inspect the physical plan — look for "BroadcastHashJoin"


**Data-parallelism notes for your video / write-up:**
- **Repartitioning** increases the number of parallel tasks (useful before a wide, CPU-heavy transformation); **coalescing** reduces partitions without a full shuffle (useful before writing output, to avoid many tiny files).
- **Caching** trades memory for recomputation time — valuable when the same DataFrame feeds multiple downstream actions (as in EDA or iterative ML training), but wasteful if a DataFrame is only used once.
- A **broadcast join** avoids shuffling the large `customer_transactions` table across the cluster network by instead copying the small lookup table to every executor — this is one of the highest-leverage optimizations in distributed joins when one side is small.
- **Trade-off:** more partitions ≠ always faster — too many partitions add scheduling overhead; too few underutilize available cores. In production this is tuned based on cluster size and data volume (`spark.sql.shuffle.partitions`, executor count, etc.).


---
## Summary & Next Steps

- **Part 1:** Built a Hive-backed data warehouse table over the banking dataset and queried it with HiveQL.
- **Part 2:** Explored the data with Spark's distributed DataFrame API — class imbalance, category-wise subscription rates, correlations.
- **Part 3:** Built and evaluated Logistic Regression and Random Forest models with Spark ML pipelines to predict term-deposit subscription.
- **Part 4:** Simulated a real-time transaction feed with Spark Structured Streaming and computed live aggregations.
- **Part 5:** Demonstrated partitioning, caching, and broadcast joins as concrete data-parallelism optimizations, with measured before/after evidence.

**Possible production extensions** (good talking points for the "Challenges & Learnings" section of your video):
- Replace the CSV-drip simulation in Part 4 with a real Kafka topic.
- Deploy on a managed cluster (EMR / Dataproc / Databricks) instead of Colab's single node.
- Add MLflow model tracking and a feature store for the Spark ML pipeline.
- Add data quality checks (e.g. Great Expectations) before writes into the Hive warehouse.
- Tune `spark.sql.shuffle.partitions` and executor sizing for real cluster hardware.
